In [ ]:
## Packages Import
%matplotlib widget
import copy
import sys
import time
import numpy             as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
from matplotlib.ticker import MaxNLocator

sys.path.append("./src/")
from VLMSurface import VLMSurface
from VLMSolver import VLMSolver

plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman']
plt.rcParams['mathtext.fontset'] = 'stix'
plt.rcParams['savefig.format'] = 'pdf'  # Put the default save format to pdf
plt.rcParams['savefig.bbox'] = 'tight'  # Crop the figure when saving


In [ ]:
## Utilities

def chord_fn(n, ar, b, sym, space, shape, lam=2.2):
    """
    Return chord length distribution along spanwise direction according to the
    desired planform shape.

    Input:
        n     -> number of spanwise stations
        ar    -> aspect ratio
        b     -> wing span
        sym   -> symmetric or not
        space -> uniform or cosine spanwise spacing
        shape -> wing planform shape, can be "rectangular", "elliptical" or "tapered"
        lam   -> taper ratio, only needed for tapered shape

    Output:
        c     -> chord length distribution along spanwise direction, shape (n,)
    """
    le = (0.5 * b)
    s_min, s_max = 0.0, le
    if space :
        if sym :
            theta = np.linspace(np.pi/2, np.pi, n)
            s = le*(-np.cos(theta))
        else :
            theta = np.linspace(0, np.pi, n)
            s = le*(1-np.cos(theta))/2
    else :
        s = np.linspace(s_min, s_max, n)
    match shape:
        case "rectangular":
            c = b/ar * np.ones(n)
        case "elliptical":
            c = 4*b/(np.pi*ar) * np.sqrt(1 - (s/le)**2)
        case "tapered":
            c = - s*4*(lam - 1)/(ar*(lam + 1)) + 4*le*lam/(ar*(lam + 1))
    return c

def plot3D(surfaces, show_mirror):
    """ 
    Plot all the surface in 3D

    Input
        surfaces    -> a list of VLMSurface
        show_mirror -> enable the display of boundary conditions 
    """
    fig_3d = plt.figure(figsize=(14, 4),constrained_layout=True)
    ax4 = fig_3d.add_subplot(111, projection='3d')
    first = {"wing":True, "wake":True, "mir_wing":True, "mir_wake":True}             # to only have one legend per item
    for surface in surfaces:
        for i, panel in enumerate(surface.wing_panels["real"]):
                pnt    = copy.copy(panel.pnt)
                vrt    = copy.copy(panel.vrt)
                # Close the polygon shape
                pnt.append(pnt[0])
                pnt_plt = np.array(pnt)
                vrt.append(vrt[0])
                vrt_plt = np.array(vrt)
                if first["wing"] :               # In order to have just one legend
                        ax4.plot(pnt_plt[:, 1], pnt_plt[:, 0], pnt_plt[:, 2], color='black', label='Wing')
                        ax4.plot(vrt_plt[:, 1], vrt_plt[:, 0], vrt_plt[:, 2], color='red', lw=0.8, label='Vortex rings')
                        first["wing"] = False
                else :
                        ax4.plot(pnt_plt[:, 1], pnt_plt[:, 0], pnt_plt[:, 2], color='black')
                        ax4.plot(vrt_plt[:, 1], vrt_plt[:, 0], vrt_plt[:, 2], color='red', lw=0.8)
        for i, panel in enumerate(surface.wake_panels["real"]):
            pnt    = copy.copy(panel.pnt)
            # Close the polygon shape
            pnt.append(pnt[0])
            pnt_plt = np.array(pnt)
            if first["wake"]:               # In order to have just one legend
                    ax4.plot(pnt_plt[:, 1], pnt_plt[:, 0], pnt_plt[:, 2], color='blue', lw=0.6, label='Wake')
                    first["wake"] = False
            else :
                    ax4.plot(pnt_plt[:, 1], pnt_plt[:, 0], pnt_plt[:, 2], color='blue', lw=0.6)
        if show_mirror:
            for i, panel in enumerate(surface.wing_panels["mirror"]):
                    pnt    = copy.copy(panel.pnt)
                    vrt    = copy.copy(panel.vrt)
                    # Close the polygon shape
                    pnt.append(pnt[0])
                    pnt_plt = np.array(pnt)
                    vrt.append(vrt[0])
                    vrt_plt = np.array(vrt)
                    if first["mir_wing"] :               # In order to have just one legend
                            ax4.plot(pnt_plt[:, 1], pnt_plt[:, 0], pnt_plt[:, 2], color='dimgray', label='Mirrored wing')
                            ax4.plot(vrt_plt[:, 1], vrt_plt[:, 0], vrt_plt[:, 2], color='forestgreen', lw=0.8, label='Mirrored vortex rings')
                            first["mir_wing"] = False
                    else :
                            ax4.plot(pnt_plt[:, 1], pnt_plt[:, 0], pnt_plt[:, 2], color='dimgray')
                            ax4.plot(vrt_plt[:, 1], vrt_plt[:, 0], vrt_plt[:, 2], color='forestgreen', lw=0.8)
            for i, panel in enumerate(surface.wake_panels["mirror"]):
                pnt    = copy.copy(panel.pnt)
                # Close the polygon shape
                pnt.append(pnt[0])
                pnt_plt = np.array(pnt)
                if first["mir_wake"]:               # In order to have just one legend
                        ax4.plot(pnt_plt[:, 1], pnt_plt[:, 0], pnt_plt[:, 2], color='darkviolet', lw=0.6, label='Mirrored wake')
                        first["mir_wake"] = False
                else :
                        ax4.plot(pnt_plt[:, 1], pnt_plt[:, 0], pnt_plt[:, 2], color='darkviolet', lw=0.6)
            
    ax4.view_init(elev=20, azim=35)
    x_lim = ax4.get_xlim3d()
    y_lim = ax4.get_ylim3d()
    z_lim = ax4.get_zlim3d()
    ax4.set_xlim(x_lim[0],x_lim[1])
    ax4.set_ylim(y_lim[0],y_lim[1]) 
    ax4.set_zlim(z_lim[1],z_lim[0])
    ax4.set_box_aspect([-x_lim[0]+x_lim[1],-y_lim[0]+y_lim[1],-z_lim[0]+z_lim[1]])
    ax4.set_xlabel("y", fontstyle='italic')
    ax4.set_ylabel("x", fontstyle='italic')
    ax4.set_zlabel("z", fontstyle='italic')
    ax4.xaxis.set_major_locator(MultipleLocator(0.5))
    ax4.yaxis.set_major_locator(MultipleLocator(1))
    ax4.zaxis.set_major_locator(MultipleLocator(0.5))

    ax4.set_title('3D view')
    ax4.legend()

def plot2D(surfaces, show_mirror, axis):
    """ 
    Plot all the surfaces in 2D

    Input
        surfaces    -> a list of VLMSurface
        show_mirror -> enable the display of boundary conditions 
        axis        -> axis that goes out - 0 for x ; 1 for y ; 2 for z
    """
    fig_2d = plt.figure(figsize=(14/1.5, 8/1.5),constrained_layout=True)
    ax4 = fig_2d.add_subplot()
    first = {"wing":True, "wake":True, "mir_wing":True, "mir_wake":True}             # to only have one legend per item
    for surface in surfaces:
        for i, panel in enumerate(surface.wing_panels["real"]):
                pnt    = copy.copy(panel.pnt)
                vrt    = copy.copy(panel.vrt)
                ctr    = copy.copy(panel.ctr)
                # Close the polygon shape
                pnt.append(pnt[0])
                pnt_plt = np.array(pnt)
                vrt.append(vrt[0])
                vrt_plt = np.array(vrt)
                if first["wing"] :               # In order to have just one legend
                        ax4.plot(pnt_plt[:, (axis+1)%3], pnt_plt[:, (axis+2)%3], color='black', label='Wing panels')
                        ax4.plot(vrt_plt[:, (axis+1)%3], vrt_plt[:, (axis+2)%3], color='red', lw=0.8, label='Vortex rings')
                        #ax4.plot(ctr[(axis+1)%3], ctr[(axis+2)%3], "o", ms=5, markerfacecolor='None', markeredgecolor='red',  label='Control points')
                        first["wing"] = False
                else :
                        ax4.plot(pnt_plt[:, (axis+1)%3], pnt_plt[:, (axis+2)%3], color='black')
                        ax4.plot(vrt_plt[:, (axis+1)%3], vrt_plt[:, (axis+2)%3], color='red', lw=0.8)
                        #ax4.plot(ctr[(axis+1)%3], ctr[(axis+2)%3], "o", ms=5, markerfacecolor='None', markeredgecolor='red')
        for i, panel in enumerate(surface.wake_panels["real"]):
            pnt    = copy.copy(panel.pnt)
            # Close the polygon shape
            pnt.append(pnt[0])
            pnt_plt = np.array(pnt)
            if first["wake"]:               # In order to have just one legend
                    ax4.plot(pnt_plt[:, (axis+1)%3], pnt_plt[:, (axis+2)%3], color='blue', lw=0.6, label='Wake panels')
                    first["wake"] = False
            else :
                    ax4.plot(pnt_plt[:, (axis+1)%3], pnt_plt[:, (axis+2)%3], color='blue', lw=0.6)
        if show_mirror:
            for i, panel in enumerate(surface.wing_panels["mirror"]):
                    pnt    = copy.copy(panel.pnt)
                    vrt    = copy.copy(panel.vrt)
                    # Close the polygon shape
                    pnt.append(pnt[0])
                    pnt_plt = np.array(pnt)
                    vrt.append(vrt[0])
                    vrt_plt = np.array(vrt)
                    if first["mir_wing"] :               # In order to have just one legend
                            ax4.plot(pnt_plt[:, (axis+1)%3], pnt_plt[:, (axis+2)%3], color='dimgray', label='Mirrored wing')
                            ax4.plot(vrt_plt[:, (axis+1)%3], vrt_plt[:, (axis+2)%3], color='forestgreen', lw=0.8, label='Mirrored vortex rings')
                            first["mir_wing"] = False
                    else :
                            ax4.plot(pnt_plt[:, (axis+1)%3], pnt_plt[:, (axis+2)%3], color='dimgray')
                            ax4.plot(vrt_plt[:, (axis+1)%3], vrt_plt[:, (axis+2)%3], color='forestgreen', lw=0.8)
            for i, panel in enumerate(surface.wake_panels["mirror"]):
                pnt    = copy.copy(panel.pnt)
                # Close the polygon shape
                pnt.append(pnt[0])
                pnt_plt = np.array(pnt)
                if first["mir_wake"]:               # In order to have just one legend
                        ax4.plot(pnt_plt[:, (axis+1)%3], pnt_plt[:, (axis+2)%3], color='darkviolet', lw=0.6, label='Mirrored wake')
                        first["mir_wake"] = False
                else :
                        ax4.plot(pnt_plt[:, (axis+1)%3], pnt_plt[:, (axis+2)%3], color='darkviolet', lw=0.6)
            
    if axis == 0 :
        ax4.invert_yaxis()
        ax4.set_xlabel("y", fontstyle='italic')
        ax4.set_ylabel("z", fontstyle='italic')
        ax4.set_title('Back view')
    elif axis == 1 :
        for line in ax4.lines:
            xdata, ydata = line.get_xdata(), line.get_ydata()
            line.set_data(ydata, xdata)
        ax4.relim()
        ax4.invert_yaxis()
        ax4.set_xlabel("x", fontstyle='italic')
        ax4.set_ylabel("z", fontstyle='italic')
        ax4.set_title('Lateral view')
    else :
        for line in ax4.lines:
            xdata, ydata = line.get_xdata(), line.get_ydata()
            line.set_data(ydata, xdata)
        ax4.set_xlabel("y", fontstyle='italic')
        ax4.set_ylabel("x", fontstyle='italic')
        ax4.relim()
        ax4.invert_yaxis()
        ax4.set_title('Top view')

    x_lim = ax4.get_xlim()
    y_lim = ax4.get_ylim()
    ax4.set_xlim(x_lim[0],x_lim[1])
    ax4.set_ylim(y_lim[0],y_lim[1]) 
    ax4.set_aspect('equal')
    

    ax4.legend()
    # =======================================================
    # FIN DE LA FONCTION : Placement robuste sans collision
    # =======================================================
    # 'adjustable=box' est crucial pour garder tes chiffres et ton échelle 100% exacts
    ax4.set_aspect('equal', adjustable='box') 
    
    ax4.xaxis.set_major_locator(MultipleLocator(0.1))
    ax4.yaxis.set_major_locator(MultipleLocator(0.1))

    # On force le titre à monter légèrement pour ne jamais être collé au cadre
    ax4.set_title(ax4.get_title(), y=1.08)

    # 1. On place la légende à l'extérieur droit de la figure
    # On utilise loc="center left" et on la décale à X = 1.05 (juste après la fin du graphique)
    ax4.legend(
        loc="center left", 
        bbox_to_anchor=(1.05, 0.5), # Centrée verticalement à droite
        borderaxespad=0,
        ncol=1 # En colonne simple pour rester propre à droite
    )

    # 2. On ajuste manuellement les marges de la figure globale (14, 8)
    # On laisse 25% d'espace vide à droite (right=0.75) pour que la légende y respire 
    # sans jamais compresser le graphique ou toucher le titre.
    fig_2d.subplots_adjust(left=0.08, right=0.75, top=0.85, bottom=0.15)
    ax4.xaxis.set_major_locator(MaxNLocator(nbins=8, prune='both'))
    ax4.yaxis.set_major_locator(MaxNLocator(nbins=6, prune='both'))



In [ ]:
## Parameters Definition

# Wing geometry
B = 1        # wing span                 [m]
AR = 1       # aspect ration             [-]
ALPHA  = 10  # angle of attack           [deg] - positive definite for counterclockwise rotations about y-axis
BETA   = 0   # drift angle               [deg] - positive definite for counterclockwise rotations about z-axis
LAMBDA = 0   # middle-chord sweep angle  [deg] - positive definite for counterclockwise rotations about x-axis  Y-AXIS
DELTA  = 0    # dihedral angle            [deg] - positive definite for rotations oriented towards positive y-axis  X-AXIS, négatif pour dyhedre classique 
PHI    = 0    # twist tip angle           [deg] - negative for washout

SYM   = True           # symmetric wing configuration
SPACE = True        # spacing distribution, False = uniform, True = cos
SHAPE = "rectangular"   # shape of the wing
TIME  = "classic"       # time step distribution - classic or cosine at the starting vortex
FREE  = False        # free surface enable

# Flow properties
U = 1.0     # inflow velocity [m/s]

# Wing discretization
N = 15      # number of panels in spanwise direction
M = 5      # number of panels in chordwise direction

# Time simulation parameters
T  = 3   # length of the simulation in s
DT = 0.1    # time step lentgh 




In [ ]:
## Test VLMSurface

# the cutoff distance is the min panel width times the ratio
ratio = 0.4        # with the ratio smaller than 0.5 the global result are accurate but the distribution is singular, above 0.5 it is the other way around. The ratio retained for the report was 0.4
a_ratio = 0.4      # for the A matrix this ratio needs to be smaller than 0.5 otherwise some control points are in the cutoff, it actually is the same for every value below 0.5. It is a parameter to stay still when the cutoff ratio is changed for other part of the code  
shedding = "both"  # wake shedding option, can be "both", "right", "left" or "none" / the issue at the tip is due to tip shedding

surface = VLMSurface(origin=np.array([0,0,0]),
                     plan=1,
                     boundary=FREE,
                     shedding=shedding,
                     b=B,
                     c=chord_fn((N+1), AR, B, SYM, SPACE, "rectangular"),
                     alpha=np.deg2rad(ALPHA),
                     beta=np.deg2rad(BETA),
                     lamb=np.deg2rad(LAMBDA),
                     delta=np.deg2rad(DELTA),
                     phi=np.deg2rad(PHI),
                     sym=SYM,
                     space=SPACE,
                     n=N,
                     m=M)
surface._build_wing()
vlm = VLMSolver([surface], np.array([U,0,0]), FREE, ratio, a_ratio)
vlm._time_sim(t=T, dt=DT, distribution=TIME)
vlm._kuttas_loads()
plot3D([surface],False)
fig = plt.figure(figsize=(14/1.5, 8/1.5),constrained_layout=True)
ax  = fig.add_subplot()
ctrl    = np.array([p.ctr for p in surface.wing_panels["real"]])
z       = np.sum(ctrl.reshape((M,2*N,3)), axis=0)[:,1]/M
z       = z/B
ax.plot(z, -surface.Cl_2d)
ax.set_xlabel("Spanwise location "+ r"$y/b$")
ax.set_ylabel("2D lift coefficient "+ r"$C_l$")
ax.set_ylim(0, np.max(-surface.Cl_2d)*1.1)


print("Experimental lift coef  :"+" 0.33 (AR=1; ALPHA=10°)")
print("Rodriguez VLM lift coef :"+" 0.37 (AR=1; ALPHA=10°)")
print("VLM lift coef           :", -2*surface.loads[2]*AR/(B**2), "(AR=",AR,"; ALPHA=",ALPHA,"°)" )


In [ ]:
## Lift coefficient versus simulation time
# Compare the two load computations (Kutta-Joukowski and pressure difference) as the wake develops
timee = np.linspace(1, 8, 8)
cl_kj = []
cl_dp = []
for ti in timee :
    surf_t = VLMSurface(origin=np.array([0,0,0]),
                        plan=1,
                        boundary=FREE,
                        shedding=shedding,
                        b=B,
                        c=chord_fn((2+1), AR, B, SYM, SPACE, SHAPE),
                        alpha=np.deg2rad(ALPHA),
                        beta=np.deg2rad(BETA),
                        lamb=np.deg2rad(LAMBDA),
                        delta=np.deg2rad(DELTA),
                        phi=np.deg2rad(PHI),
                        sym=SYM,
                        space=SPACE,
                        n=2, m=2)
    surf_t._build_wing()
    vlm_t = VLMSolver([surf_t], np.array([U, 0.0, 0.0]), FREE, ratio, a_ratio)
    vlm_t._time_sim(ti, DT, distribution=TIME)
    vlm_t._kuttas_loads()
    cl_kj.append(-2*surf_t.loads[2]*AR/(B**2))
    vlm_t._secondary_computation()
    cl_dp.append(-2*surf_t.loads[2]*AR/(B**2))
print("time [s]            :", timee)
print("CL Kutta-Joukowski  :", np.round(cl_kj, 4))
print("CL pressure         :", np.round(cl_dp, 4))
